### **Standared Prompt**

In [ ]:
OPEN_AI_API_KEY="voc-3231998021403751658726a2fc7d2440498.60015628"
OPEN_AI_BASE_URL="https://openai.vocareum.com/v1"

In [ ]:
import os
import json
from openai import OpenAI

# 1. Initialize the OpenAI client
# It will automatically look for the OPENAI_API_KEY environment variable.
client = OpenAI(api_key=OPEN_AI_API_KEY,
    base_url=OPEN_AI_BASE_URL)

def extract_transaction_data(user_input: str) -> dict:
    """
    Takes a natural language string and extracts transaction data using OpenAI.
    """

    # 2. Define the structured system prompt
    # Note: When using OpenAI's JSON mode, you MUST instruct the model to return JSON.
    system_instruction = """
    ROLE: You are an expert financial data extraction assistant specialized in trading and currency exchange.

    TASK:
    1. Analyze the user's natural language input.
    2. Extract four core fields:
      - "intent": The action type ("buy", "sell", or "convert").
      - "amount": The numerical amount (as a standard float or integer).
      - "source_currency": The standard ticker symbol or ISO code of the asset being sold/exchanged (e.g., "BTC", "USD", "EUR").
      - "target_currency": The standard ticker symbol or ISO code of the asset being acquired, or null if unspecified.
    3. Normalize currency names and slang to official tickers (e.g., "tether" -> "USDT", "bucks" -> "USD", "ether" -> "ETH").
    4. Gracefully correct common typos (e.g., "but" -> "buy", "sel" -> "sell").
    5. If the target currency is not specified, set "target_currency" to null.

CONTEXT: Users provide informal, chat-based, or voice-transcribed instructions to trade, exchange, or convert cryptocurrencies and fiat currencies.

EXAMPLES:


    FORMAT: Return ONLY a valid JSON object with the exact keys: "intent", "amount", "source_currency", and "target_currency".
    """

    try:
        # 3. Make the API call using gpt-4o-mini (fast and cost-effective)
        response = client.chat.completions.create(
            model="gpt-4o-mini",

            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": user_input}
            ],

            # This forces the model to output valid JSON
            response_format={"type": "json_object"},
            temperature=0.0 # Set to 0 for highly deterministic, consistent data extraction
        )

        # 4. Extract and parse the JSON string from the response
        raw_json_string = response.choices[0].message.content
        extracted_data = json.loads(raw_json_string)

        return extracted_data

    except json.JSONDecodeError:
        print("Error: Failed to parse JSON.")
        return {}
    except Exception as e:
        print(f"An API error occurred: {e}")
        return {}

# ==========================================
# Run the Program
# ==========================================
if __name__ == "__main__":
    # Test Case 1: Your example with the typo "but"
    test_input_1 = "I want to but 3000 BTC to CAD"
    print(f"Processing: '{test_input_1}'")
    result_1 = extract_transaction_data(test_input_1)

    print("\nExtracted Data Dictionary:")
    print(json.dumps(result_1, indent=4))

    print("-" * 40)

    # Test Case 2: A different variation
    test_input_2 = "Can you sell 50.5 Ethereum for US Dollars please?"
    print(f"Processing: '{test_input_2}'")
    result_2 = extract_transaction_data(test_input_2)

    print("\nExtracted Data Dictionary:")
    print(json.dumps(result_2, indent=4))


Processing: 'I want to but 3000 BTC to CAD'

Extracted Data Dictionary:
{
    "intent": "buy",
    "amount": 3000,
    "source_currency": "BTC",
    "target_currency": "CAD"
}
----------------------------------------
Processing: 'Can you sell 50.5 Ethereum for US Dollars please?'

Extracted Data Dictionary:
{
    "intent": "sell",
    "amount": 50.5,
    "source_currency": "ETH",
    "target_currency": "USD"
}


## **Few shot samples**

In [ ]:
import os
import json
from openai import OpenAI

# 1. Initialize the OpenAI client
# It will automatically look for the OPENAI_API_KEY environment variable.
client = OpenAI(api_key=OPEN_AI_API_KEY,
    base_url=OPEN_AI_BASE_URL)

def extract_transaction_data(user_input: str) -> dict:
    """
    Takes a natural language string and extracts transaction data using OpenAI.
    """

    # 2. Define the structured system prompt
    # Note: When using OpenAI's JSON mode, you MUST instruct the model to return JSON.
    system_instruction = """
    ROLE: You are an expert financial data extraction assistant specialized in trading and currency exchange.

TASK:
1. Analyze the user's natural language input.
2. Extract four core fields:
   - "intent": The action type ("buy", "sell").
   - "amount": The numerical amount (as a standard float or integer).
   - "source_currency": The standard ticker symbol or ISO code of the asset being sold/exchanged (e.g., "BTC", "USD", "EUR").
   - "target_currency": The standard ticker symbol or ISO code of the asset being acquired, or null if unspecified.
   - "confidance score" : how confidant you are in extracted entities give a value between 0-1
3. Normalize currency names and slang to official tickers (e.g., "tether" -> "USDT", "bucks" -> "USD", "ether" -> "ETH").
4. Gracefully correct common typos (e.g., "but" -> "buy", "sel" -> "sell").
5. If the target currency is not specified, set "target_currency" to null.
6. If the target currency is not specified, set "USD".

CONTEXT: Users provide informal, chat-based, or voice-transcribed instructions to trade, exchange, or convert cryptocurrencies and fiat currencies.

EXAMPLES:

# Example 1: Typo correction & crypto-to-fiat conversion
User Input: "I want to but 3000 BTC to CAD"
Output:
{
  "intent": "buy",
  "amount": 3000,
  "source_currency": "BTC",
  "target_currency": "CAD"
}

# Example 2: Fiat-to-crypto conversion
User Input: "Convert 500 US dollars to bitcoin"
Output:
{
  "intent": "Sell",
  "amount": 500,
  "source_currency": "USD",
  "target_currency": "BTC"
}

# Example 3: Selling crypto for fiat with decimal amounts
User Input: "Selling 0.45 ETH for euros"
Output:
{
  "intent": "sell",
  "amount": 0.45,
  "source_currency": "ETH",
  "target_currency": "EUR"
}

# Example 4: Fiat-to-fiat exchange with number formatting (commas)
User Input: "Exchange 1,500 GBP into Japanese Yen"
Output:
{
  "intent": "convert",
  "amount": 1500,
  "source_currency": "GBP",
  "target_currency": "JPY"
}

# Example 5: Crypto-to-crypto trade with slang/ticker shorthand
User Input: "Swap 100k doge to tether"
Output:
{
  "intent": "convert",
  "amount": 100000,
  "source_currency": "DOGE",
  "target_currency": "USDT"
}

# Example 6: Single-direction purchase (Missing target/payment currency)
User Input: "I want to buy 15 Solana"
Output:
{
  "intent": "buy",
  "amount": 15,
  "source_currency": "SOL",
  "target_currency": null
}

# Example 7: Slang liquidation / selling
User Input: "Dump 2.5 etherium"
Output:
{
  "intent": "sell",
  "amount": 2.5,
  "source_currency": "ETH",
  "target_currency": null
}

FORMAT: Return ONLY a valid JSON object matching the key names: "intent", "amount", "source_currency", "target_currency", and confidance_score. Do not include Markdown blocks or extra commentary.
    """

    try:
        # 3. Make the API call using gpt-4o-mini (fast and cost-effective)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": user_input}
            ],
            # This forces the model to output valid JSON
            response_format={"type": "json_object"},
            temperature=0.0 # Set to 0 for highly deterministic, consistent data extraction
        )

        # 4. Extract and parse the JSON string from the response
        raw_json_string = response.choices[0].message.content
        extracted_data = json.loads(raw_json_string)

        return extracted_data

    except json.JSONDecodeError:
        print("Error: Failed to parse JSON.")
        return {}
    except Exception as e:
        print(f"An API error occurred: {e}")
        return {}

# ==========================================
# Run the Program
# ==========================================
if __name__ == "__main__":
    # Test Case 1: Your example with the typo "but"
    test_input_1 = "I want to but 3000 BTC to CAD"
    print(f"Processing: '{test_input_1}'")
    result_1 = extract_transaction_data(test_input_1)

    print("\nExtracted Data Dictionary:")
    print(json.dumps(result_1, indent=4))

    print("-" * 40)

    # Test Case 2: A different variation
    test_input_2 = "Can you sell 50.5 Ethereum for US Dollars please?"
    print(f"Processing: '{test_input_2}'")
    result_2 = extract_transaction_data(test_input_2)

    print("\nExtracted Data Dictionary:")
    print(json.dumps(result_2, indent=4))

    # Test Case 2: A different variation
    test_input_2 = "can i exchage 30000 bitcoin for Indian Rupee?"
    print(f"Processing: '{test_input_2}'")
    result_2 = extract_transaction_data(test_input_2)

    print("\nExtracted Data Dictionary:")
    print(json.dumps(result_2, indent=4))

Processing: 'I want to but 3000 BTC to CAD'

Extracted Data Dictionary:
{
    "intent": "buy",
    "amount": 3000,
    "source_currency": "BTC",
    "target_currency": "CAD",
    "confidance_score": 0.9
}
----------------------------------------
Processing: 'Can you sell 50.5 Ethereum for US Dollars please?'

Extracted Data Dictionary:
{
    "intent": "sell",
    "amount": 50.5,
    "source_currency": "ETH",
    "target_currency": "USD",
    "confidance_score": 1.0
}
Processing: 'can i exchage 30000 bitcoin for Indian Rupee?'

Extracted Data Dictionary:
{
    "intent": "convert",
    "amount": 30000,
    "source_currency": "BTC",
    "target_currency": "INR",
    "confidance_score": 0.9
}


In [ ]:
"""You are an expert financial and treasury data extraction agent. Your task is to extract core foreign exchange (FX) transaction entities from unstructured text, notices, or trade confirmations.

To ensure extreme accuracy with multi-currency values and exchange rates, you must follow a strict Chain-of-Thought reasoning process before producing the final JSON output.

### Instructions:
1. **Analyze the Input:** Read the raw FX text message or trade confirmation carefully.
2. **Step 1 - Determine Action & Direction:** Identify whether the transaction is a Buy, Sell, Transfer, or Spot/Forward conversion from the perspective of the account holder.
3. **Step 2 - Extract Financial Entities:** Identify the key attributes:
   - Source/Sold Currency & Amount (what is leaving or being converted)
   - Target/Bought Currency & Amount (what is being received)
   - Foreign Exchange Rate (and whether it's direct or indirect quote)
   - Value Date / Trade Date
   - Counterparty / Intermediary Bank / Platform
   - Reference ID or UETR (if available)
4. **Step 3 - Normalize & Validate:** Standardize all currency codes to ISO 4217 (e.g., USD, EUR, GBP, INR). Verify that: `Source Amount * Exchange Rate = Target Amount` (allowing for standard spread or fee deviations). Format dates to ISO 8601 (YYYY-MM-DD).
5. **Step 4 - Format Output:** Output your step-by-step reasoning, followed strictly by the extracted data in valid JSON format.

---
### Example:
Input: "CONFIRMATION: You sold 10,000.00 USD and bought 845,200.00 INR at an all-in FX rate of 84.52 on 22-Aug-2026. Ref: FX-9928104."

Reasoning Process:
- Step 1: The transaction is a currency exchange (Sell USD, Buy INR).
- Step 2: Source Currency = USD, Source Amount = 10000.00, Target Currency = INR, Target Amount = 845200.00, Exchange Rate = 84.52, Trade Date = 22-Aug-2026, Reference = FX-9928104.
- Step 3: Normalizing currencies to "USD" and "INR". Validating calculation: 10,000 * 84.52 = 845,200.00 (Math matches). Date formatted to "2026-08-22".
- Step 4: Ready for JSON mapping.

JSON Output:
{
  "transaction_type": "FX Conversion",
  "action": "Sell USD / Buy INR",
  "source_currency": "USD",
  "source_amount": 10000.00,
  "target_currency": "INR",
  "target_amount": 845200.00,
  "exchange_rate": 84.52,
  "trade_date": "2026-08-22",
  "reference_id": "FX-9928104",
  "status": "Confirmed"
}

---
### Your Turn:
Input: "{{USER_FX_TRANSACTION_TEXT}}"

Reasoning Process:"""